# AIMO 3 — R1-TIR Submission (LoRA Fine-Tuned)

**DeepSeek-R1-Distill-Qwen-32B + LoRA SFT** on NuminaMath-TIR.

- Long chain-of-thought reasoning + code execution
- LoRA fine-tuned for tool-integrated reasoning
- Majority voting across 8-16 candidates
- Auto-tunes to H100 (80GB) or GH200 (96GB)
- **Score: 5/10 on IMO-level reference problems**

In [ ]:
# Dependencies are pre-installed on Kaggle GPU images or via utility script.
# If running interactively with internet, uncomment:
# import subprocess, sys
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "vllm", "transformers", "polars"])

In [ ]:
import os, warnings
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"  # fix protobuf MessageFactory error
warnings.filterwarnings("ignore")

import re, subprocess, sys, tempfile, time
from collections import Counter
from dataclasses import dataclass
from typing import Optional
import pandas as pd
import polars as pl
import torch

IS_KAGGLE = os.path.exists("/kaggle")
IS_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

# Time management
CUTOFF_TIME = time.time() + (4 * 60 + 45) * 60  # stop 15 min before 5hr limit
PER_PROBLEM_TIMEOUT = 360  # 6 min max per problem (50 problems × 6 min = 5 hrs)
problem_count = 0

@dataclass
class Config:
    model_id: str = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"
    # H100 (80GB): use 16K context (was 10K — too short), fewer samples
    # With 16K context on H100: ~2x concurrency, enough for 4 samples batched
    num_samples: int = 4 if VRAM_GB < 90 else 16
    num_rounds: int = 3
    max_tokens_per_round: int = 12288 if VRAM_GB < 90 else 16384
    max_model_len: int = 16384 if VRAM_GB < 90 else 32768
    gpu_mem_util: float = 0.95 if VRAM_GB < 90 else 0.92
    temperature: float = 0.6
    top_p: float = 0.95
    code_timeout: int = 8

CFG = Config()
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
print(f"VRAM: {VRAM_GB:.0f}GB | samples={CFG.num_samples} | max_tok={CFG.max_tokens_per_round} | ctx={CFG.max_model_len}")

In [ ]:
# ============= Code Execution =============

IMPORTS = """\
import math, sys
import numpy as np
import sympy as sp
from sympy import *
from fractions import Fraction
from itertools import permutations, combinations, product
from functools import reduce
from collections import Counter, defaultdict
"""
FORBIDDEN = {"subprocess", "os.system", "__import__", "shutil", "pathlib"}

def execute_code(code, timeout=10):
    for tok in FORBIDDEN:
        if tok in code:
            return False, f"{tok} blocked"
    src = IMPORTS + "\n" + code
    lines = src.strip().split("\n")
    last = lines[-1].strip()
    skip = ("print","#","import","from","def ","class ","if ","for ","while ",
            "try","with ","else","elif","except","finally","return","raise",
            "assert","pass","break","continue","@")
    if last and not last.startswith(skip) and "=" not in last:
        lines[-1] = f"print({last})"
        src = "\n".join(lines)
    with tempfile.TemporaryDirectory() as td:
        p = os.path.join(td, "s.py")
        with open(p, "w") as f:
            f.write(src)
        try:
            r = subprocess.run([sys.executable, p], capture_output=True, text=True, timeout=timeout, check=False)
            if r.returncode == 0:
                out = r.stdout.strip()
                return True, out[:1200] + ("..." if len(out) > 1200 else "")
            err = r.stderr.strip().split("\n")
            return False, err[-1] if err else "error"
        except subprocess.TimeoutExpired:
            return False, f"Timed out ({timeout}s)"
        except Exception as e:
            return False, str(e)

def last_code_block(text):
    blocks = re.findall(r"```python\s*\n(.*?)```", text, re.DOTALL)
    return blocks[-1] if blocks else None

In [ ]:
# ============= Answer Extraction =============

def extract_boxed(text):
    i = text.rfind("\\boxed{")
    if i < 0: return None
    start = i + 7
    depth, j = 1, start
    while j < len(text) and depth > 0:
        if text[j] == "{": depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0: return text[start:j]
        j += 1
    return None

def to_int(s):
    if not s: return None
    s = s.strip()
    for r in ("\\text{","\\mathrm{","\\,",","," ","$","}","\\"):
        s = s.replace(r, "")
    try: return int(s)
    except: pass
    try: return round(float(s))
    except: pass
    if "/" in s:
        try:
            from fractions import Fraction
            f = Fraction(s)
            if f.denominator == 1: return f.numerator
        except: pass
    m = re.search(r"-?\d+", s)
    return int(m.group(0)) if m else None

In [ ]:
# ============= Lazy Model Loader =============
# Model must be lazy-loaded: serve() must be called within 15 min of startup

try:
    from vllm import LLM, SamplingParams
except AttributeError:
    # Suppress protobuf MessageFactory error on Kaggle
    from vllm import LLM, SamplingParams

class LazyModel:
    def __init__(self):
        self.llm = None
        self.tokenizer = None

    def load(self):
        model_path = CFG.model_id
        if IS_KAGGLE:
            candidates = [
                "/kaggle/input/datasets/tantheta/r1-tir-merged",
                "/kaggle/input/r1-tir-merged",
                "/kaggle/input/datasets/tantheta/r1-distill-qwen-32b",
                "/kaggle/input/r1-distill-qwen-32b",
            ]
            for candidate in candidates:
                if os.path.exists(candidate):
                    model_path = candidate
                    break
        print(f"Loading {model_path}...", flush=True)
        t0 = time.time()
        self.llm = LLM(
            model=model_path,
            dtype="bfloat16",
            gpu_memory_utilization=CFG.gpu_mem_util,
            max_model_len=CFG.max_model_len,
            enforce_eager=False,
            trust_remote_code=True,
        )
        self.tokenizer = self.llm.get_tokenizer()
        print(f"Loaded in {time.time()-t0:.0f}s", flush=True)

    def get(self):
        if self.llm is None:
            self.load()
        return self.llm, self.tokenizer

model = LazyModel()

In [ ]:
# ============= R1-TIR Solver =============

R1_PROMPT = """{problem}

Solve this step-by-step. When you need to compute something, write Python code in ```python blocks. \
After each code block, I will run it and show you the output. \
Put your final integer answer in \\boxed{{}}."""

def build_prompt(tokenizer, problem):
    messages = [{"role": "user", "content": R1_PROMPT.format(problem=problem)}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def solve(problem_text):
    global problem_count
    problem_count += 1

    # Return a guess if running out of time
    if time.time() > CUTOFF_TIME:
        print("  CUTOFF: returning 0", flush=True)
        return 0

    problem_start = time.time()

    try:
        llm, tokenizer = model.get()
        base = build_prompt(tokenizer, problem_text)
        base_len = len(tokenizer.encode(base))
        max_prompt_tokens = CFG.max_model_len - 1024
        N = CFG.num_samples
        suffixes = [""] * N
        done = [False] * N
        stops = ["```output", "```\n\n", "<|im_end|>", "<|endoftext|>", "</think>"]

        for rnd in range(CFG.num_rounds):
            # Per-problem timeout check
            if time.time() - problem_start > PER_PROBLEM_TIMEOUT:
                print(f"  Problem timeout ({PER_PROBLEM_TIMEOUT}s), extracting best answer so far", flush=True)
                break

            live = [i for i in range(N) if not done[i]]
            if not live: break

            prompts, batch_live = [], []
            for i in live:
                plen = base_len + len(tokenizer.encode(suffixes[i])) if suffixes[i] else base_len
                if plen >= max_prompt_tokens:
                    done[i] = True
                    continue
                prompts.append(base + suffixes[i])
                batch_live.append(i)
            live = batch_live
            if not live: break

            sp = SamplingParams(
                temperature=CFG.temperature, top_p=CFG.top_p,
                max_tokens=CFG.max_tokens_per_round,
                stop=stops, include_stop_str_in_output=True,
                seed=rnd * 1000 + 42, n=1,
            )
            outs = llm.generate(prompts, sp, use_tqdm=False)

            for j, out in enumerate(outs):
                i = live[j]
                text = out.outputs[0].text
                suffixes[i] += text
                fr = out.outputs[0].finish_reason
                sr = str(out.outputs[0].stop_reason or "")

                if extract_boxed(suffixes[i]) is not None:
                    done[i] = True; continue
                if sr == "</think>": continue

                triggered = sr in ("```output", "```\n\n") or (
                    fr == "stop" and suffixes[i].rstrip().endswith("```")
                    and suffixes[i].count("```python") > suffixes[i].count("```output"))
                if triggered:
                    code = last_code_block(suffixes[i])
                    if code:
                        ok, res = execute_code(code, CFG.code_timeout)
                        res = res[:600]
                        if sr == "```output":
                            suffixes[i] += f"\n{res}\n```\n"
                        else:
                            sfx = "" if suffixes[i].endswith("\n") else "\n"
                            suffixes[i] += f"{sfx}```output\n{res}\n```\n"
                    continue
                if fr == "length": continue
                if fr == "stop": done[i] = True

        # Force answer for unfinished candidates
        still_open = [i for i in range(N) if not done[i] and extract_boxed(suffixes[i]) is None]
        if still_open and (time.time() - problem_start) < PER_PROBLEM_TIMEOUT:
            fps = [base + suffixes[i] + "\n\nThe final answer is \\boxed{" for i in still_open]
            sp = SamplingParams(temperature=0.0, max_tokens=64, stop=["}"], include_stop_str_in_output=True, n=1)
            outs = llm.generate(fps, sp, use_tqdm=False)
            for j, out in enumerate(outs):
                suffixes[still_open[j]] += "\n\nThe final answer is \\boxed{" + out.outputs[0].text

        # Extract all answers and vote
        answers = []
        for i in range(N):
            boxed = extract_boxed(suffixes[i])
            val = to_int(boxed) if boxed else None
            if val is not None and 0 <= val <= 99999:
                answers.append(val)
        if answers:
            return Counter(answers).most_common(1)[0][0]
        return 0
    except Exception as e:
        print(f"  ERROR in solve: {e}", flush=True)
        return 0

In [ ]:
# ============= Submission (matches official demo pattern) =============

import kaggle_evaluation.aimo_3_inference_server

def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    """Predict function matching the official AIMO3 API signature.
    Args match the columns of test.csv unpacked via *data_batch."""
    id_val = id_.item(0)
    problem_text = question.item(0)
    t0 = time.time()
    pred = solve(problem_text)
    print(f"  [{id_val}] -> {pred} ({time.time()-t0:.0f}s)", flush=True)
    return pl.DataFrame({"id": id_val, "answer": pred})

inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv",)
    )